# 🩺 MONAI Label Server with Cloudflare Tunnel

This notebook sets up and runs a **MONAI Label** server in Google Colab, integrated with Google Drive for persistent storage. It also uses **Cloudflare Tunnel** to provide a secure, public URL so you can connect your local medical imaging software (like 3D Slicer) directly to this cloud-hosted AI server.

### 🚀 Key Features:
*   **MONAI Label Server:** AI-assisted labeling for medical imaging.
*   **Cloudflare Tunnel:** Exposes the server without complex port forwarding.
*   **Google Drive Integration:** Saves your data and trained models to `/MyDrive/cancer_ai`.
*   **Automated Setup:** Handles dependency conflicts (NumPy/pydicom) automatically.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

### 📂 Project Directory Setup
This section defines the base directory in your Google Drive and creates the necessary subfolders for data, models, and logs. This ensures your project structure is organized and persistent.

In [4]:
import os

BASE_DIR = "/content/drive/MyDrive/cancer_ai"

folders = [
    "data",
    "models",
    "transforms",
    "logs",
    "apps"
]

for f in folders:
    os.makedirs(os.path.join(BASE_DIR, f), exist_ok=True)

print("Folders created under:", BASE_DIR)

Folders created under: /content/drive/MyDrive/cancer_ai


### 🛠 Environment Installation
This cell handles the installation of specific versions of NumPy and pydicom to avoid compatibility issues. It also installs the MONAI Label core package and downloads the Cloudflare tunnel binary.

In [ ]:
print("🔧 Setting up environment...")

# ---- 1. Fix NumPy version first ----
!pip uninstall -y numpy >/dev/null 2>&1
!pip install -q numpy==1.26.4

# ---- 2. Install pydicom versions required for compatibility ----
!pip uninstall -y pydicom >/dev/null 2>&1
!pip install -q pydicom==2.4.4
!pip install -q pydicom-seg==0.4.1

# ---- 3. Install core dependencies ----
!pip install -q antspyx==0.4.2
!pip install -q monailabel

# ---- 4. Cloudflare tunnel ----
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

import os
os.environ["CANCER_AI_ENV_READY"] = "1"

print("\n✅ Installation complete")
print("⚠️ IMPORTANT: Restart Runtime now (Runtime → Restart runtime)")

### 🔍 Environment Verification
After restarting the runtime, run this cell to verify that the core libraries like NumPy and ANTs are correctly loaded. It ensures that the environment is stable before starting the server.

In [2]:
# =========================================
# Environment Check
# Run this cell *after* restart the runtime
# =========================================

import numpy as np
import ants
import monailabel

print("NumPy:", np.__version__)
print("✅ Environment is working correctly")

NumPy: 1.26.4
✅ הכל עובד


### 🧬 MONAI Label App Selection
This section clones the official MONAI Label repository to access sample applications. We will specifically use the radiology app for this setup.

In [10]:
# This cell is now merged into the main setup block above.
print("✅ pydicom configuration moved to setup cell")

Found existing installation: pydicom 3.0.2
Uninstalling pydicom-3.0.2:
  Successfully uninstalled pydicom-3.0.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 24.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pynetdicom 3.0.4 requires pydicom<4,>=3, but you have pydicom 2.4.4 which is incompatible.
✅ pydicom fixed


In [7]:
!git clone https://github.com/Project-MONAI/MONAILabel.git

Cloning into 'MONAILabel'...
remote: Enumerating objects: 15212, done.
remote: Counting objects: 100% (59/59), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 15212 (delta 23), reused 8 (delta 7), pack-reused 15153 (from 4)
Receiving objects: 100% (15212/15212), 55.00 MiB | 27.74 MiB/s, done.
Resolving deltas: 100% (10595/10595), done.


### 🚀 Server Startup
This cell initializes the MONAI Label server in the background using `nohup`. It points the server to your Google Drive folders so that any progress is automatically saved.

In [ ]:
import os

DATA_DIR = os.path.join(BASE_DIR, "data")
MODEL_DIR = os.path.join(BASE_DIR, "models")



In [23]:
import os
import subprocess

# Configuration
STUDIES_DIR = "/content/drive/MyDrive/cancer_ai/data"
MODEL_DIR = "/content/drive/MyDrive/cancer_ai/models"
APP_DIR = "/content/MONAILabel/sample-apps/radiology"

# Ensure directories exist
os.makedirs(STUDIES_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# Create a dummy file if empty to prevent server crash
if not os.listdir(STUDIES_DIR):
    with open(os.path.join(STUDIES_DIR, ".keep"), "w") as f:
        f.write("")

print("🚀 Starting MONAI Label server in background...")

# Run server in background and redirect output to a log file
get_ipython().system(f'nohup monailabel start_server --app "{APP_DIR}" --studies "{STUDIES_DIR}" --conf models segmentation --conf model_dir "{MODEL_DIR}" --port 8000 > server.log 2>&1 &')

import time
time.sleep(5)
print("✅ Server is starting. Check server.log for details.")

🚀 Starting MONAI Label server in background...
✅ Server is starting. Check server.log for details.


### 🌐 Secure Tunneling
This script runs the Cloudflare tunnel to expose the local server to the internet. It will monitor the logs and display the unique public URL you need to enter into 3D Slicer.

In [25]:
import subprocess
import time
import re

print("🌍 Starting Cloudflare tunnel...")

# Start cloudflared
proc = subprocess.Popen(
    ["./cloudflared-linux-amd64", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# Look for the URL in the output stream
url_found = False
for line in proc.stdout:
    # print(line.strip()) # Uncomment to see full debug logs
    if "trycloudflare.com" in line:
        url = re.search(r"https://[\w-]+\.trycloudflare\.com", line)
        if url:
            print("\n" + "="*30)
            print("🔗 YOUR MONAI LABEL URL:")
            print(url.group(0))
            print("="*30)
            url_found = True
            break

if not url_found:
    print("❌ Could not find the tunnel URL. Please check if the server in the previous cell is actually running.")

🌍 Starting Cloudflare tunnel...

🔗 YOUR MONAI LABEL URL:
https://bugs-reference-journalism-baseline.trycloudflare.com


### 🛑 Shutdown
Use this cell when you are finished to safely stop both the MONAI Label server and the Cloudflare tunnel. This clears the background processes and releases the system resources.

In [26]:
import os
import subprocess

print("🛑 Shutting down MONAI Label server and Cloudflare tunnel...")

# Kill monailabel and cloudflared processes
!pkill -f monailabel
!pkill -f cloudflared

print("✅ Processes stopped.")

🛑 Shutting down MONAI Label server and Cloudflare tunnel...
✅ Processes stopped.


In [ ]:
--conf models_dir=/content/drive/MyDrive/cancer_ai/models